# ALS-Based Movie Recommendation Engine

### - Project Objective

The goal of this project is to build a personalized movie recommendation system using collaborative filtering techniques on Databricks Free Edition.

The system predicts:

“Which movies is a user most likely to rate highly?”

This is achieved using the ALS (Alternating Least Squares) algorithm from Apache Spark MLlib.

## Full Pipeline Architecture

Bronze Layer → Raw CSV data

Silver Layer → Cleaned & structured data

ML Layer → ALS model training

Evaluation Layer → RMSE performance validation

Recommendation Layer → Manual generation of top-N movies

Presentation Layer → Join with movie titles




#🥉 Bronze Layer (Data Ingestion)
Load MovieLens datasets:
Ratings (userId, movieId, rating)
Movies (movieId, title, genres)
Store raw data as-is
No cleaning or transformation
Schema inferred automatically














In [0]:
# Load ratings data
ratings_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/movie_volume/ratings.csv")

# Load movies data
movies_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/movie_volume/movies.csv")

ratings_df.show(5)
movies_df.show(5)


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


#🥈 Silver Layer (Data Cleaning)
Select required columns: userId, movieId, rating
Convert data types:
userId, movieId → int
rating → float
Remove null values

✔ Output: Clean, structured data ready for ML

In [0]:
from pyspark.sql.functions import col

ratings_clean = ratings_df \
    .select(
        col("userId").cast("int"),
        col("movieId").cast("int"),
        col("rating").cast("float")
    ) \
    .dropna()

movies_clean = movies_df \
    .select(
        col("movieId").cast("int"),
        col("title"),
        col("genres")
    ) \
    .dropna()

ratings_clean.printSchema()
movies_clean.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



#🤖 Model Building (ALS)
Use ALS (collaborative filtering)

Learns:
User preferences
Movie features

Key configs:
userCol, itemCol, ratingCol
coldStartStrategy = "drop"
nonnegative = True

🧪 Train-Test Split
80% training
20% testing

✔ Ensures proper evaluation on unseen data

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Split data
train, test = ratings_clean.randomSplit([0.8, 0.2])

# Define ALS model
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

# Train
model = als.fit(train)

# Predict
predictions = model.transform(test)


#📊 Model Evaluation
Metric: RMSE

Lower RMSE = better accuracy

In [0]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)


RMSE: 0.8785821119029494


#🎯 Recommendation Generation (Manual)
(Used due to Free Edition limitations)

- Get all users
- Get all movies
- Create all user–movie pairs
- Predict ratings using ALS
- Rank movies per user
- Select Top 5 recommendations per user

In [0]:
users = ratings_clean.select("userId").distinct()
movies_df = ratings_clean.select("movieId").distinct()

from pyspark.sql.functions import col

user_movie = users.crossJoin(movies_df)



In [0]:
predictions = model.transform(user_movie)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("userId").orderBy(col("prediction").desc())

top_recommendations = predictions.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") <= 5)

top_recommendations.show()


+------+-------+----------+----+
|userId|movieId|prediction|rank|
+------+-------+----------+----+
|     1|   3925| 5.8375907|   1|
|     1|   3379|  5.782273|   2|
|     1|  33649| 5.6698594|   3|
|     1|  93988| 5.6432405|   4|
|     1| 171495| 5.5921564|   5|
|     2|  84847| 5.1726904|   1|
|     2|  33649| 5.0417128|   2|
|     2|  93988|  4.952152|   3|
|     2|   7669| 4.9358606|   4|
|     2| 131724| 4.8994193|   5|
|     3|  70946| 5.0461216|   1|
|     3|   5746|  4.890913|   2|
|     3|   6835|  4.890913|   3|
|     3|   5181| 4.8420897|   4|
|     3|   7991| 4.8237925|   5|
|     4|   6818|  5.160552|   1|
|     4|  89904|  5.136283|   2|
|     4|   6375| 5.1212473|   3|
|     4|  27611| 5.0390444|   4|
|     4| 148881| 4.9306617|   5|
+------+-------+----------+----+
only showing top 20 rows


In [0]:
final = top_recommendations.join(movies_clean, "movieId")

final.select("userId", "title", "prediction").show(truncate=False)


+------+------------------------------------------------------------------------------+----------+
|userId|title                                                                         |prediction|
+------+------------------------------------------------------------------------------+----------+
|15    |Pledge, The (2001)                                                            |4.9861755 |
|15    |Drugstore Cowboy (1989)                                                       |4.9166236 |
|15    |Children of Dune (2003)                                                       |4.819124  |
|15    |Frozen River (2008)                                                           |4.819124  |
|15    |Dune (2000)                                                                   |4.753606  |
|20    |Stranger Than Paradise (1984)                                                 |5.322361  |
|20    |Great Escape, The (1963)                                                      |5.3033047 |
|20    |Tr

#🚀 Final Result

A complete pipeline that:

- Ingests raw data
- Cleans & prepares it
- Trains ML model
- Evaluates accuracy
- Generates personalized recommendations